In [6]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [7]:
import sys
sys.path.insert(0, "/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/diffusers/src")
from diffusers.replica_exchange.acceptance import _k_ladder

from diffusers import StableDiffusion3Pipeline
import numpy as np

In [8]:
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir="/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/model_checkpoints"
)
pipe = pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [9]:
swap_algorithm={
		"n_replicas": 3, 
		"p_ratio": "p",
		"even_indices": [0, 7, 12, 16, 19, 21],   # t ≈ 870, 763, 648, 536
		"odd_indices":  [1, 8, 13, 17, 20, 22],
		"debug": True
	}

tsr_sigma = 3.0
# labels = ["chocolates"]
# prompts = ["A box of chocolates"]
labels = ["chocolates", "badminton", "princess", "buildings", "canoe"]
prompts = ["A box of chocolates", "Boy playing badminton with his grandfather", "The Princess and the Frog Read-Along W/CD [With Paperback Book]", "London from the Sky Garden Photographic Print", "Canoe on Elk Lake"]

In [10]:
tsr_k_vals = [1.0]
replica_exchanges = [True, False]

for i, prompt in enumerate(prompts):

	for tsr_k in tsr_k_vals:
		for replica_exchange in replica_exchanges:

			generator = torch.Generator(device="cuda").manual_seed(42)

			all_images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=30,
				guidance_scale=5.0,
				tsr_k=tsr_k,
				tsr_sigma=tsr_sigma,
				replica_exchange=replica_exchange,
				swap_algorithm=swap_algorithm,
				generator=generator,
			).images
			

			for idx in range(len(all_images)):
				image = all_images[idx]
				arr = np.array(image)
				print(f"Image {idx}: min={arr.min()}, max={arr.max()}, mean={arr.mean():.2f}")

				if replica_exchange:
					k_ladder = _k_ladder(torch.tensor(tsr_k), swap_algorithm["n_replicas"], device="cpu", dtype=torch.float32)				
					k_val = k_ladder[idx]
					string = f"pt_tsr_{tsr_k}_val_{k_val:.2f}"
				else:
					string = f"tsr_{tsr_k}"

				output_path = f"/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/{labels[i]}_{string}.png"
				image.save(output_path)
				print(f"Saved to {output_path}")

We will be running with replica swaps with 3 replicas


  0%|          | 0/30 [00:00<?, ?it/s]

 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 1000.0
Time 1000.0 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.985
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 988.2713623046875
Time 988.2713623046875 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.973
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 975.9791870117188
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 963.08203125
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 949.5339965820312
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 935.2844848632812
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 920.2777099609375
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 904.4515380859375
Time 904.4515380859375 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.896
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 887.737060546875
Time 887.737060546875 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.882
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 8

  0%|          | 0/30 [00:00<?, ?it/s]

 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
Image 0: min=0, max=255, mean=145.47
Saved to /n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/chocolates_tsr_1.0.png
We will be running with replica swaps with 3 replicas


...

 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
Image 0: min=0, max=255, mean=160.62
Saved to /n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/buildings_tsr_1.0.png
We will be running with replica swaps with 3 replicas


  0%|          | 0/30 [00:00<?, ?it/s]

 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 1000.0
Time 1000.0 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.985
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 988.2713623046875
Time 988.2713623046875 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.973
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 975.9791870117188
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 963.08203125
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 949.5339965820312
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 935.2844848632812
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 920.2777099609375
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 904.4515380859375
Time 904.4515380859375 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.892
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 887.737060546875
Time 887.737060546875 swap btwn source 1.00 and target 1.00 accept 1.000 std 0.877
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
t is 8

  0%|          | 0/30 [00:00<?, ?it/s]

 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
 We tsr by 1.00
Image 0: min=0, max=255, mean=134.74
Saved to /n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/canoe_tsr_1.0.png
